# Seizure Detector — Portable Inference

This notebook loads `seizure_detector_model.keras` and applies it to an EDF EEG recording.

It reproduces the training preprocessing pipeline:

1. Load EDF
2. Keep the first 18 channels
3. Band-pass filter: 0.5–100 Hz
4. 60 Hz notch filter
5. Per-channel z-score normalization
6. Clip to ±5
7. Create 4-second windows with 50% overlap
8. Transpose each window to `(1024, 18)`
9. Predict seizure probability using the saved CNN–BiLSTM–Attention model

The notebook also creates **multimodal-fusion-ready outputs**:

- `eeg_predictions.csv` — late-fusion features and timestamps
- `eeg_fusion_features.npz` — 32-dimensional EEG embedding per window plus timestamps/probability

For multimodal fusion, use the same `SESSION_ID` and time base for EEG, face, pose, and motion streams.

> Important: the original EEG model was trained using the first 18 EDF channels rather than an explicit saved channel-name mapping. Cross-dataset deployment should therefore validate/re-map the channel montage before relying on predictions.


In [ ]:
# ============================================
# CELL 1: Imports and Environment Check
# ============================================

import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyedflib
import matplotlib.pyplot as plt
import tensorflow as tf

from scipy import signal
from scipy.signal import butter, filtfilt
from tensorflow.keras.models import load_model
from tensorflow.keras import Model

print("✅ Imports successful")
print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

if os.name == "nt":
    print("⚠️ Native Windows Python detected.")
    print("   For NVIDIA TensorFlow GPU use, run this project in WSL2 or Docker.")
else:
    print("✅ Linux/WSL/Docker Python environment detected")


In [ ]:
# ============================================
# CELL 2: Configuration
# ============================================

PROJECT_ROOT = Path.cwd()

MODEL_PATH = Path(
    os.environ.get(
        "SEIZURE_MODEL_PATH",
        str(PROJECT_ROOT / "seizure_detector_model.keras")
    )
)

METADATA_PATH = PROJECT_ROOT / "model_metadata.json"  # optional

# Change this path to the EDF file you want to test.
# You can also set EEG_EDF_PATH as an environment variable.
EDF_PATH = Path(
    os.environ.get(
        "EEG_EDF_PATH",
        str(
            PROJECT_ROOT
            / "Input"
            / "EEG Database"
            / "chb-mit-scalp-eeg-database-1.0.0"
            / "chb01"
            / "chb01_03.edf"
        )
    )
)

# Shared session ID for multimodal fusion.
# Set the SAME value for EEG/video/pose/motion outputs from one patient/session.
SESSION_ID = os.environ.get("SESSION_ID", EDF_PATH.stem)

# Values matching the training pipeline
FS = 256
N_CHANNELS = 18

WINDOW_SIZE = 4
WINDOW_SAMPLES = WINDOW_SIZE * FS

OVERLAP = 0.5
STEP_SAMPLES = int(WINDOW_SAMPLES * (1 - OVERLAP))

THRESHOLD = 0.5

OUTPUT_CSV = PROJECT_ROOT / "eeg_predictions.csv"
OUTPUT_NPZ = PROJECT_ROOT / "eeg_fusion_features.npz"

print("Model:", MODEL_PATH)
print("EDF:", EDF_PATH)
print("Session ID:", SESSION_ID)
print("Window samples:", WINDOW_SAMPLES)
print("Step samples:", STEP_SAMPLES)


In [ ]:
# ============================================
# CELL 3: Optional Metadata
# ============================================

metadata = None

if METADATA_PATH.exists():
    with METADATA_PATH.open("r", encoding="utf-8") as f:
        metadata = json.load(f)

    print("✅ Metadata loaded")
    print(json.dumps(metadata, indent=2))
else:
    print("ℹ️ No model_metadata.json found. Using notebook configuration.")


In [ ]:
# ============================================
# CELL 4: Load Saved Model
# ============================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}\n"
        "Place seizure_detector_model.keras in the project directory "
        "or set the SEIZURE_MODEL_PATH environment variable."
    )

model = load_model(MODEL_PATH, compile=False)

print("✅ Model loaded")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

expected_input = tuple(model.input_shape[1:])

if expected_input != (WINDOW_SAMPLES, N_CHANNELS):
    raise ValueError(
        "Model/input mismatch: expected inference data shaped "
        f"({WINDOW_SAMPLES}, {N_CHANNELS}), "
        f"but model reports {expected_input}."
    )

print("✅ Model input shape matches training pipeline")


In [ ]:
# ============================================
# CELL 5: Preprocessing Pipeline
# ============================================

class EEGPreprocessor:
    """Same preprocessing used during training."""

    def __init__(
        self,
        sampling_rate=256,
        low_freq=0.5,
        high_freq=100,
        notch_freq=60
    ):
        self.fs = sampling_rate
        self.low_freq = low_freq
        self.high_freq = high_freq
        self.notch_freq = notch_freq

    def bandpass_filter(self, data, order=4):
        nyquist = self.fs / 2

        low = self.low_freq / nyquist
        high = min(self.high_freq / nyquist, 0.99)

        b, a = butter(order, [low, high], btype="band")

        if data.ndim == 1:
            return filtfilt(b, a, data)

        return np.array([
            filtfilt(b, a, channel)
            for channel in data
        ])

    def notch_filter(self, data, Q=30):
        nyquist = self.fs / 2
        freq = self.notch_freq / nyquist

        b, a = signal.iirnotch(freq, Q)

        if data.ndim == 1:
            return filtfilt(b, a, data)

        return np.array([
            filtfilt(b, a, channel)
            for channel in data
        ])

    def normalize(self, data):
        if data.ndim == 1:
            return (
                (data - np.mean(data))
                / (np.std(data) + 1e-8)
            )

        return np.array([
            (channel - np.mean(channel))
            / (np.std(channel) + 1e-8)
            for channel in data
        ])

    def remove_artifacts(self, data, threshold=5):
        return np.clip(data, -threshold, threshold)

    def preprocess(self, data):
        data = self.bandpass_filter(data)
        data = self.notch_filter(data)
        data = self.normalize(data)
        data = self.remove_artifacts(data)
        return data


preprocessor = EEGPreprocessor(sampling_rate=FS)

print("✅ Preprocessor ready")
print(f"Band-pass: {preprocessor.low_freq}-{preprocessor.high_freq} Hz")
print(f"Notch: {preprocessor.notch_freq} Hz")
print("Normalization: per-channel z-score")
print("Clipping: ±5")


In [ ]:
# ============================================
# CELL 6: EDF Loader
# ============================================

def load_edf_file(file_path):
    """
    Load an EDF file and close it safely.

    Returns
    -------
    signals : ndarray
        Shape (channels, samples)
    channel_names : list[str]
    sampling_rates : list[float]
    """

    reader = None

    try:
        reader = pyedflib.EdfReader(file_path)

        n_channels = reader.signals_in_file
        channel_names = reader.getSignalLabels()

        sampling_rates = [
            float(reader.getSampleFrequency(i))
            for i in range(n_channels)
        ]

        signals = [
            reader.readSignal(i)
            for i in range(n_channels)
        ]

        min_length = min(len(ch) for ch in signals)

        signals = np.array(
            [ch[:min_length] for ch in signals],
            dtype=np.float32
        )

        return signals, channel_names, sampling_rates

    finally:
        if reader is not None:
            reader.close()


print("✅ EDF loader ready")


In [ ]:
# ============================================
# CELL 7: Prepare EDF for Inference
# ============================================

if not EDF_PATH.exists():
    raise FileNotFoundError(
        f"EDF file not found: {EDF_PATH}\n"
        "Update EDF_PATH in Cell 2 or set EEG_EDF_PATH."
    )

signals, channel_names, sampling_rates = load_edf_file(str(EDF_PATH))

print("✅ EDF loaded")
print("Raw shape:", signals.shape)
print("Number of EDF channels:", len(channel_names))
print("First 18 channels:")

for i, name in enumerate(channel_names[:N_CHANNELS], start=1):
    print(f"  {i:2d}. {name}")

if signals.shape[0] < N_CHANNELS:
    raise ValueError(
        f"Model requires {N_CHANNELS} channels, "
        f"but EDF contains only {signals.shape[0]}."
    )

used_sampling_rates = sampling_rates[:N_CHANNELS]

if not all(np.isclose(rate, FS) for rate in used_sampling_rates):
    raise ValueError(
        "The first 18 channels do not all use the expected "
        f"{FS} Hz sampling rate. Found: {used_sampling_rates}"
    )

# Same channel selection used during training.
signals = signals[:N_CHANNELS, :]

duration_sec = signals.shape[1] / FS

print(f"Duration: {duration_sec:.1f} seconds")
print(f"Using shape: {signals.shape}")


In [ ]:
# ============================================
# CELL 8: Preprocess Entire Recording
# ============================================

# Training preprocessing was applied to the entire EDF
# before the recording was split into windows.
signals_clean = preprocessor.preprocess(signals)

print("✅ Preprocessing complete")
print("Input shape :", signals.shape)
print("Output shape:", signals_clean.shape)
print(
    "Clean range:",
    f"[{signals_clean.min():.3f}, {signals_clean.max():.3f}]"
)


In [ ]:
# ============================================
# CELL 9: Create Model Windows
# ============================================

def create_inference_windows(signals_clean):
    windows = []
    start_samples = []

    n_samples = signals_clean.shape[1]

    for start in range(
        0,
        n_samples - WINDOW_SAMPLES + 1,
        STEP_SAMPLES
    ):
        end = start + WINDOW_SAMPLES

        window = signals_clean[:, start:end]

        if window.shape != (N_CHANNELS, WINDOW_SAMPLES):
            continue

        # Training Cell 12:
        # (18, 1024) -> (1024, 18)
        window = window.T.astype(np.float32)

        windows.append(window)
        start_samples.append(start)

    if not windows:
        raise ValueError(
            "No valid windows could be created from this EDF file."
        )

    X_infer = np.stack(windows, axis=0)

    return X_infer, np.array(start_samples)


X_infer, start_samples = create_inference_windows(signals_clean)

print("✅ Windows created")
print("Model input array shape:", X_infer.shape)
print("Number of windows:", len(X_infer))

if tuple(X_infer.shape[1:]) != tuple(model.input_shape[1:]):
    raise ValueError(
        f"Prepared data shape {X_infer.shape[1:]} does not match "
        f"model input shape {model.input_shape[1:]}."
    )


In [ ]:
# ============================================
# CELL 10: Predict All Windows
# ============================================

raw_predictions = model.predict(
    X_infer,
    batch_size=32,
    verbose=1
)

seizure_probability = np.asarray(raw_predictions).reshape(-1)

if len(seizure_probability) != len(X_infer):
    raise ValueError(
        "Unexpected model output shape: "
        f"{np.asarray(raw_predictions).shape}"
    )

predicted_class = (
    seizure_probability >= THRESHOLD
).astype(np.int32)

start_sec = start_samples / FS
end_sec = (start_samples + WINDOW_SAMPLES) / FS

# Stable time-based IDs for alignment with video/pose/motion outputs.
window_ids = [
    f"{SESSION_ID}_{int(round(s * 1000)):010d}_{int(round(e * 1000)):010d}"
    for s, e in zip(start_sec, end_sec)
]

results = pd.DataFrame({
    "session_id": SESSION_ID,
    "window_id": window_ids,
    "start_sec": start_sec,
    "end_sec": end_sec,
    "eeg_seizure_probability": seizure_probability,
    "eeg_predicted_class": predicted_class,
    "eeg_predicted_label": np.where(
        predicted_class == 1,
        "Seizure",
        "Normal"
    )
})

print("\n✅ Prediction complete")
display(results.head(10))


In [ ]:
# ============================================
# CELL 11: Prediction Summary
# ============================================

n_total = len(results)
n_seizure = int((results["eeg_predicted_class"] == 1).sum())
n_normal = n_total - n_seizure

print("=" * 60)
print("📊 INFERENCE SUMMARY")
print("=" * 60)
print(f"File: {EDF_PATH}")
print(f"Session: {SESSION_ID}")
print(f"Total windows: {n_total}")
print(f"Normal windows: {n_normal}")
print(f"Seizure windows: {n_seizure}")
print(f"Decision threshold: {THRESHOLD:.2f}")
print(
    "Maximum predicted seizure probability:",
    f"{results['eeg_seizure_probability'].max():.2%}"
)

if n_seizure > 0:
    print("\n🚨 Windows classified as seizure:")
    display(
        results.loc[
            results["eeg_predicted_class"] == 1,
            [
                "window_id",
                "start_sec",
                "end_sec",
                "eeg_seizure_probability",
                "eeg_predicted_label"
            ]
        ].reset_index(drop=True)
    )
else:
    print("\n✅ No windows crossed the seizure threshold.")


In [ ]:
# ============================================
# CELL 12: Plot Seizure Probability Timeline
# ============================================

plt.figure(figsize=(14, 5))

window_midpoint = (
    results["start_sec"] + results["end_sec"]
) / 2

plt.plot(
    window_midpoint,
    results["eeg_seizure_probability"],
    linewidth=1
)

plt.axhline(
    THRESHOLD,
    linestyle="--",
    label=f"Threshold = {THRESHOLD}"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Predicted seizure probability")
plt.title(f"EEG Seizure Probability — {SESSION_ID}")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# ============================================
# CELL 13: Export Fusion-Ready Outputs
# ============================================

# 1) Late-fusion output: timestamped EEG probabilities
results.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Saved prediction table: {OUTPUT_CSV}")
print(f"Rows written: {len(results)}")

# 2) Intermediate-fusion output:
# The penultimate Dense layer ("dense_1") has a 32-D representation.
try:
    embedding_layer = model.get_layer("dense_1")
except ValueError as exc:
    raise ValueError(
        "Expected penultimate layer 'dense_1' was not found. "
        "The uploaded model architecture may have changed."
    ) from exc

embedding_model = Model(
    inputs=model.input,
    outputs=embedding_layer.output
)

eeg_embeddings = embedding_model.predict(
    X_infer,
    batch_size=32,
    verbose=1
).astype(np.float32)

if eeg_embeddings.ndim != 2 or eeg_embeddings.shape[0] != len(results):
    raise ValueError(
        f"Unexpected EEG embedding shape: {eeg_embeddings.shape}"
    )

np.savez_compressed(
    OUTPUT_NPZ,
    session_id=np.asarray([SESSION_ID] * len(results)),
    window_id=results["window_id"].to_numpy(),
    start_sec=results["start_sec"].to_numpy(dtype=np.float64),
    end_sec=results["end_sec"].to_numpy(dtype=np.float64),
    eeg_seizure_probability=results[
        "eeg_seizure_probability"
    ].to_numpy(dtype=np.float32),
    eeg_embedding=eeg_embeddings
)

print(f"✅ Saved EEG fusion features: {OUTPUT_NPZ}")
print("EEG embedding shape:", eeg_embeddings.shape)
print("\nFor late fusion, use eeg_predictions.csv.")
print("For intermediate fusion, use eeg_fusion_features.npz.")


## Notes

- The sigmoid output is treated as a **predicted seizure probability**, with `0.5` used as the current decision threshold. It is not necessarily a calibrated clinical probability.
- Inference intentionally reproduces the original training preprocessing, including a 60 Hz notch filter and whole-recording per-channel normalization.
- The original model selected the **first 18 EDF channels**, not an explicit saved channel-name montage. Before using a different EEG dataset, verify or map channel identities/order.
- `eeg_predictions.csv` is designed for **late fusion**.
- `eeg_fusion_features.npz` contains a 32-D EEG representation from the penultimate Dense layer for **intermediate fusion**.
- For multimodal alignment, face/body/motion pipelines should use the same `session_id`, and their output should either use the same 4-second windows / 2-second stride or be aligned by timestamp overlap.
- The original training procedure used a window-level split. For research evaluation, rebuild training/validation/test splits by patient/session to avoid leakage from overlapping windows.
